In [ ]:
'''

The code is the based on the "Mutiple Loci and Multiple Dimension" in 181006_3. Testing of Universal pleiotropy. We further 
collect the statistics of mutations as that in 1.1. code. The collected mutation statistics includes:

1. Mean effect of new generating mutations in each generation within each population 
2. Mean absolute effect of new generating mutations in each generation within each population
(also, each of their effects is also collected and stored)

3. Mean effect of all existing mutations within each population
4. Mean absolute effect of all existing mutations within each population

5. Mean Proportion of beneficial mutations among all new generated mutations in each generation.
6. Mean effect of beneficial mutations generated each generation

7. The overall proportion of beneficial mutations within each population(i.e., for all existing mutations)
8. The overall mean effect of beneficial mutations within each populations(i.e., for all existing mutations)

'''

In [ ]:
from __future__ import division
import numpy as np
from scipy import stats
import scipy.spatial as spa
import numpy.random as rnd
import copy
import time
import pandas as pd
import math
import pickle
import multiprocessing as mp

In [ ]:
class Population(object):
    '''Define a single population in FGM. Assuming the fixed optimum is at the orgin of the Coordinate System'''

    def __init__(self,size, nLoci, dim,fit,mut_rate,sigma, ploidy =1):
        
        self.initial_fitness = fit  # The initial fitness of the population

        self.nLoci = nLoci
        self.dim = dim  # The dimension of the landscape
        self.size = size # The population size 
        self.mutation_rate = mut_rate/(dim*ploidy) # mutation rate
        self.sigma = sigma/np.sqrt(dim)  # The STD of generating new mutations
        
        self.ploidy = ploidy  # The ploidy of the individual within the population
        
        self.soma_mut_moving_pos = np.zeros((size, nLoci, ploidy, dim)) # These two np.array will be used to store how much that each mutation in each locus in
        self.germ_mut_moving_pos = np.zeros((size, nLoci, 2, dim))     # Soma and Germ will move along the corresponding dimension. Initially no mutations
        
        self.sel_coeff = np.zeros((size, dim, ploidy)) # Will be used to store the mutation effect of each mutation                                                                # Thus the values within the array are 0.

        self.pop_mean_fit = []

        self.beneficial_prop = []
        self.beneficial_s_mean = []

        self.s_mean = []  # Mean effect of new generating mutations in each generation within each population
        self.s_abs_mean = [] # Mean abs effect of new generating mutations in each generation within each population

        self.mut_s = []  # The effect of all the new generating mutations in each generation within each population
        self.bene_mut_s = [] # The effect of all the new generating beneficial mutations in each generation within each population


        self.exist_mut_s_mean = []  # Mean effect of all existing mutations within each population
        self.exist_mut_abs_s_mean = [] # Mean abs effect of all existing mutations within each population

        self.exist_mut_bene_prop = []  # The overall prop of beneficial mutations within each population
        self.exist_mut_bene_s_mean = []


        self.generate_ancestor()  # Generate ancestors with pre-defined initial fitness
        self.create_pop()  # Create populations composed of ancestors
        
        self.current_step =0  # Will be used to indicate the number of generations that have been simulated.



    
        
    def calculate_pop_fitness(self): 
        '''Calculate the fitness of each individual within the whole population simultaneously'''
        
        euclidean_distance = np.sqrt(np.sum((self.pop_point-np.zeros((self.size,self.dim)))**2, axis =1))      
        fitness = np.exp(-(euclidean_distance**2)/2)
        return fitness


    @staticmethod
    def calculate_ind_fitness(point):
        '''Calculate the fitness of a point'''
        euclidean_distance = np.sqrt(np.sum((point)**2))
        fitness = np.exp(-(euclidean_distance**2)/2)
        return fitness

    
    def generate_ancestor(self): 
        '''Generate ancestor with the predefined initial fitness'''
   
        if self.initial_fitness == 1: #special case if want ancestor at optimum
            self.anc_point = np.zeros(self.dim) #place ancestor at optimum
            self.anc_fit = 1
        else:
            self.anc_point = np.random.normal(size=self.dim,scale=.01) 
            self.anc_fit = np.exp(-(spa.distance.euclidean(self.anc_point,np.zeros(self.dim))**2)/2) 
            scaling = 1 


            if self.anc_fit < self.initial_fitness: 

                while self.anc_fit < self.initial_fitness: 
                    scaling = scaling + .0001 #increase scalar
                    self.anc_point = self.anc_point/scaling 
                    self.anc_fit = np.exp(-(spa.distance.euclidean(self.anc_point,np.zeros(self.dim))**2)/2) #test fitness

            elif self.anc_fit > self.initial_fitness: 

                while self.anc_fit > self.initial_fitness:  #keep moving ancestor away from optimum until desired initial fitness reached
                    scaling = scaling + .0001 #increase scalar
                    self.anc_point = self.anc_point * scaling 
                    self.anc_fit = np.exp(-(spa.distance.euclidean(self.anc_point,np.zeros(self.dim))**2)/2)            
            

            
    def create_pop(self):
        """ Create population - initially monomorphic. """

        self.anc_pop_point = np.repeat([self.anc_point], self.size, axis =0)
        
        self.pop_point = np.repeat([self.anc_point], self.size, axis =0)  
        self.pop_fit = np.repeat([float(self.anc_fit)], self.size, axis =0)
        
        self.soma_site = np.ones((self.size, self.nLoci, self.ploidy), dtype ='int')
        self.germ_site = np.ones((self.size, self.nLoci, 2), dtype ='int')

        
        
    def mutate(self):
        '''Generate new mutations in both Soma and Germ'''
        
        # Use the new method introduced by Ricardo.
        
        # How many new mutations are generated
        soma_mut_occur = np.random.binomial(self.soma_site, self.mutation_rate)
        germ_mut_occur = np.random.binomial(self.germ_site, self.mutation_rate)

        soma_mut_num = len(soma_mut_occur[soma_mut_occur == 1])
        germ_mut_num = len(germ_mut_occur[germ_mut_occur == 1])



        
        # Then deal with the moving of each new mutation
        self.mut_matrix = np.identity(self.dim)*(self.sigma**2) #isotropic mutation matrix w/ no covariance

        soma_mut_move = rnd.multivariate_normal(mean=np.zeros(self.dim),cov=self.mut_matrix,size= soma_mut_num)
        germ_mut_move = rnd.multivariate_normal(mean=np.zeros(self.dim),cov=self.mut_matrix,size=germ_mut_num)
        


        if self.current_step%100 ==0:

            temporary_soma_mut_moving_pos = self.soma_mut_moving_pos.copy()
            temporary_soma_mut_moving_pos[soma_mut_occur ==1] = soma_mut_move


            previous_soma_mut_moving_pos = self.soma_mut_moving_pos.copy() # Moving positions of previous mutations
            previous_fit = self.pop_fit.copy() # Population fitness before this mutation occur
            previous_mfit = np.nanmean(previous_fit)


            new_gen_s = []
            new_gen_bene_s = []

            for i in range(self.size):
                for j in range(self.dim):
                    for k in range(self.ploidy):
                        if soma_mut_occur[i][j][k] !=0:
                            previous_point = self.pop_point[i].copy()
                            previous_point_fit = self.pop_fit[i].copy()

                            new_point = previous_point.copy()

                            new_point = previous_point- previous_soma_mut_moving_pos[i][j][k] \
                            + temporary_soma_mut_moving_pos[i][j][k]

                            new_point_fit = self.calculate_ind_fitness(new_point)

                            sel_coeff = (new_point_fit - previous_point_fit)/previous_mfit

                            self.sel_coeff[i][j][k] =sel_coeff

                            new_gen_s.append(sel_coeff)

                            if sel_coeff >0:
                                new_gen_bene_s.append(sel_coeff)


            bene_prop = [len(new_gen_s), len(new_gen_bene_s)]
            self.beneficial_prop.append(bene_prop)

            new_bene_mut_mean_s = np.nanmean(new_gen_bene_s)

            self.bene_mut_s.append(new_gen_bene_s)
            self.beneficial_s_mean.append(new_bene_mut_mean_s)


            self.mut_s.append(new_gen_s)
            new_gen_mut_mean_s = np.nanmean(new_gen_s)
            self.s_mean.append(new_gen_mut_mean_s)

            new_gen_mut_abs_mean_s = np.nanmean(np.abs(new_gen_s))
            self.s_abs_mean.append(new_gen_mut_abs_mean_s)


            all_mut_mean_s = np.nanmean(self.sel_coeff[self.sel_coeff!=0])

            abs_sel_coeff = abs(self.sel_coeff)
            all_mut_abs_mean_s = np.nanmean(abs_sel_coeff[abs_sel_coeff!=0])

            self.exist_mut_s_mean.append(all_mut_mean_s)
            self.exist_mut_abs_s_mean.append(all_mut_abs_mean_s)

            exist_mut_bene_prop = [len(self.sel_coeff[self.sel_coeff!=0]), len(self.sel_coeff[self.sel_coeff>0])]
            self.exist_mut_bene_prop.append(exist_mut_bene_prop)

            exist_mut_bene_s_mean = np.nanmean(self.sel_coeff[self.sel_coeff>0])
            self.exist_mut_bene_s_mean.append(exist_mut_bene_s_mean)



        self.soma_mut_moving_pos[soma_mut_occur ==1] = soma_mut_move
        self.germ_mut_moving_pos[germ_mut_occur ==1] = germ_mut_move

        soma_moving_pos_each_loci = np.sum(self.soma_mut_moving_pos, axis =2)
        soma_moving_pos = np.sum(soma_moving_pos_each_loci, axis =1)

        self.pop_point = self.anc_pop_point + soma_moving_pos
        self.pop_fit = self.calculate_pop_fitness()        


       
    def selection(self):
        '''Select the parents for next generation based on their relative fitness. Select with replacement.'''

        assert self.pop_fit.min() >= 0 # ensure that the fitness of every one within the population is non-negative.
        
        total_w = np.sum(self.pop_fit)   # First caclulate the total fitness within the population
        relfit = self.pop_fit/total_w  # Calculate the relative fitness of each individual
        csrelfit = np.cumsum(relfit)
        randvals = np.random.random(self.size)
        parents = np.searchsorted(csrelfit,randvals)
        
        return parents
    
    

    def mitosis(self, parents):

        '''Asexual reproduction by mitosis. Just get an copy of the selected parents (inclduing the position, the fitness of the population, and
        the moving position of mutations in both Soma and Germ .'''

             
        self.soma_mut_moving_pos = self.soma_mut_moving_pos[parents]
        self.germ_mut_moving_pos = self.germ_mut_moving_pos[parents]
        
        # Then based on the new soma_moving_pos, recalculate the pop_point and pop_fit

        soma_moving_pos_each_loci = np.sum(self.soma_mut_moving_pos, axis =2)
        soma_moving_pos = np.sum(soma_moving_pos_each_loci, axis =1)

        self.pop_point = self.anc_pop_point + soma_moving_pos
        self.pop_fit = self.calculate_pop_fitness()


      
    
    def amitosis(self, parents):
        '''Asexual reproduction by amitosis.'''
        
        self.soma_mut_moving_pos = self.soma_mut_moving_pos[parents] # First get the moving positions of mutations in Soma of selected parents
        new_soma_mut_moving_pos = np.repeat(self.soma_mut_moving_pos, 2, axis =2) # Duplicate the mutations in Soma of selected parents
        
        # Make sure that shuffle can actually do the shuffling. Maybe try random.choice first.
        for i in range(self.size):
            for j in range(self.nLoci):
                np.random.shuffle(new_soma_mut_moving_pos[i][j])  # Shuffle the duplicated mutations in Soma
                
        self.soma_mut_moving_pos = new_soma_mut_moving_pos[:, :, :self.ploidy]  # Pick the first self.ploidy mutations as the mutations in offspring Soma
        
        soma_moving_pos_each_loci = np.sum(self.soma_mut_moving_pos, axis =2)
        soma_moving_pos = np.sum(soma_moving_pos_each_loci, axis =1)

        self.pop_point = self.anc_pop_point + soma_moving_pos
        self.pop_fit = self.calculate_pop_fitness() 
       
        self.germ_mut_moving_pos = self.germ_mut_moving_pos[parents]   # Get a copy of the Germ mutations in selected parents
        
        
        
    def asexual_reproduction(self,asex_type='amitosis'):
        """Take populations through a single Wright-Fisher model generation"""

        self.mutate()
        parents = self.selection()
        if asex_type == 'amitosis':
            self.amitosis(parents)
        elif asex_type == 'mitosis':
            self.mitosis(parents)
        self.current_step += 1

    


    def get_results(self):
        '''Get soma data from the simulation'''

        W = self.pop_fit


        mW = np.nanmean(W)
        log_mW = np.log(mW)

        varW = np.var(W)


        # pop_mean_fit = np.nanmean(W, axis =1)  # Calculate mean fitness from another way: first get the mean fitness of each population

        # self.pop_mean_fit.append(pop_mean_fit)
            
        return [mW, log_mW, varW]



    def simulate(self,stepcount, asex_type='amitosis',sex_freq=None,random_mating=False,strides=10):

        '''Run the simulation for certain generations with defined reproduction strategy.'''
        
        """stepcount - total number of generations or time steps to run
        asex_type - type of asexual reproduction: mitosis, amitosis
        sex_freq - frequency of sexual reproduction
        random_mating - selfing or random mating
        strides - how frequently to get data using the get_results method
        
        returns results as a list of lists"""


        results = [self.get_results()]

        start = time.time()

        while self.current_step <= stepcount:
            if (sex_freq != None) and (self.current_step%sex_freq == 0):
                self.sex(random_mating)
            else:
                self.asexual_reproduction(asex_type)


            if self.current_step%strides == 0:
                results.append(self.get_results())


        colnames = ['meanFit','ln_meanFit','varFit']

        results = pd.DataFrame(np.array(results),columns=colnames)

        # overall_pop_m_fit = self.pop_mean_fit

        overall_pop_m_fit = self.pop_mean_fit
        
        overall_bene_prop = self.beneficial_prop
        overall_bene_s_mean = self.beneficial_s_mean

        overall_s_mean = self.s_mean
        overall_s_abs_mean = self.s_abs_mean


        overall_sel_coeff = self.sel_coeff
        overall_mut_s = self.mut_s
        overall_bene_mut_s = self.bene_mut_s
        overall_exist_mut_s_mean = self.exist_mut_s_mean
        overall_exist_mut_abs_s_mean = self.exist_mut_abs_s_mean
        overall_exist_mut_bene_prop = self.exist_mut_bene_prop
        overall_exist_mut_bene_s_mean = self.exist_mut_bene_s_mean

        end = time.time()
        print 'TOTAL TIME: ',end-start
        return results, overall_pop_m_fit, overall_bene_prop, overall_bene_s_mean, overall_s_mean, overall_s_abs_mean, overall_sel_coeff, \
        overall_mut_s, overall_bene_mut_s, overall_exist_mut_s_mean, overall_exist_mut_abs_s_mean, overall_exist_mut_bene_prop, \
        overall_exist_mut_bene_s_mean






def save_object(obj, filename):
    with open(filename, 'wb') as output:
        pickle.dump(obj, output, pickle.HIGHEST_PROTOCOL)



def get_result(data_list, data_name):

    '''Get the data from the stored data_list according to their name'''

    n = len(data_list)
    mean_data = []

    for i in range(n):
        mean_data.append(list(data_list[i][data_name]))

    m = len(list(data_list[0][data_name]))
    # print 'M', m

    each_gen_mean_data = []

    each_gen_std_data = [] # Need to check

    for j in range(m):
        each_gen_data = []
        for s in range(n):
            each_gen_data.append(mean_data[s][j])

        each_gen_mean_data.append(np.nanmean(each_gen_data))
        each_gen_std_data.append(np.nanstd(each_gen_data))  # Need check
  
    return each_gen_mean_data, each_gen_std_data



def transform_to_gen_data(replicate_data):

    rep = len(replicate_data)
    generation = len(replicate_data[0])

    generation_data = []

    for i in range(generation):
        each_gen_data = []
        for j in range(rep):
            each_gen_data.append(replicate_data[j][i])

        generation_data.append(each_gen_data)

    return generation_data





def worker(n):
    '''define a function worker to conduct a single task. Here the worker was used to run a single replicate for the
    whole simulation process. Here for I am using object-oriented programming, to let the code run, the function worker 
    is used to run the object. The function worker should have parameters otherwise it will raise error when running. But 
    the parameter n here won't be used later (here n is just to make sure that the function has parameters)'''

    np.random.seed()
    
    a =Population(size =500, nLoci = 10, dim =10, fit =0.5, mut_rate =0.1, sigma =0.05, ploidy =2)

    results = a.simulate(2000, asex_type='amitosis',sex_freq=None,random_mating=False,strides=1)

    return results

In [ ]:
if __name__ == "__main__":  # add this line can make sure that multiprocess can be tested in Windows PC. 
                    

    start = time.time()

    pool = mp.Pool(2)   # creat a multiprocessing pool with 4 CPU (should be identical to the number of requested
                           # CPU in the .sh file, or = requested CPU-1)

    print 'CPU COUNT', mp.cpu_count()  
        
    results = pool.map_async(worker, list(range(100))) # most time consuming part, use parallel computing
                                                # Here map_async is asynchronical function, and what this function do
                                                # is to map asynchronical function to our defined function worker.
                                                # list(range(10)) is the parameters used for worker, and here it means run
                                                # worker for 10 times (i.e., the number of replicates)


    results = results.get()  # get the results
 
    fitness = [i[0] for i in results]
    total_pop_m_fit = [i[1] for i in results]
    total_bene_prop = [i[2] for i in results]
    total_bene_s_mean = [i[3] for i in results]
    total_s_mean = [i[4] for i in results]
    total_s_abs_mean = [i[5] for i in results]

    # sel_coeff_matrix = [i[6] for i in results]

    t_mut_s = [i[7] for i in results]
    t_bene_mut_s = [i[8] for i in results]
    t_exist_mut_s_mean = [i[9] for i in results]
    t_exist_mut_abs_s_mean = [i[10] for i in results]
    t_exist_mut_bene_prop = [i[11] for i in results]
    t_exist_mut_bene_s_mean = [i[12] for i in results]



    pop_mfit_gen_data = transform_to_gen_data(total_pop_m_fit)
    beneficial_prop_gen_data = transform_to_gen_data(total_bene_prop)
    beneficial_s_mean_gen_data = transform_to_gen_data(total_bene_s_mean)
    s_mean_gen_data = transform_to_gen_data(total_s_mean)
    s_abs_mean_gen_data = transform_to_gen_data(total_s_abs_mean)

    t_mut_s_gen_data = transform_to_gen_data(t_mut_s)
    t_bene_mut_s_gen_data = transform_to_gen_data(t_bene_mut_s)
    t_exist_mut_s_mean_gen_data = transform_to_gen_data(t_exist_mut_s_mean)
    t_exist_mut_abs_s_mean_gen_data = transform_to_gen_data(t_exist_mut_abs_s_mean)
    t_exist_mut_bene_prop_gen_data = transform_to_gen_data(t_exist_mut_bene_prop)
    t_exist_mut_bene_s_mean_gen_data = transform_to_gen_data(t_exist_mut_bene_s_mean)


 
    gen_mfit = get_result(fitness, 'meanFit')
    gen_mfit_mean = gen_mfit[0]
    gen_mfit_std = gen_mfit[1]

    gen_mlogfit = get_result(fitness, 'ln_meanFit') 
    gen_mlogfit_mean = gen_mlogfit[0]
    gen_mlogfit_std = gen_mlogfit[1]  

    gen_varfit = get_result(fitness, 'varFit')
    gen_varfit_mean = gen_varfit[0]
    gen_varfit_std = gen_varfit[1]

     
    fitness_result = []

    for j in range(len(gen_mfit_mean)):

        fitness_result.append([gen_mfit_mean[j], gen_mfit_std[j], gen_mlogfit_mean[j], gen_mlogfit_std[j], \
            gen_varfit_mean[j], gen_varfit_std[j]])



    colnames = ['mFit_Mean','mFit_Std', 'mlogFit_Mean','mlogFit_Std','varFit_Mean', 'varFit_Std']


    fitness_result = pd.DataFrame(np.array(fitness_result),columns=colnames)

    fitness_result.to_csv('UvPley_Dim10_P2_Amito_Fit_N500_Fit05_SM005.csv', index =False)

    save_object(pop_mfit_gen_data, 'UvPley_Dim10_P2_Amito_PopMFit_N500_Fit05_SM005')
    save_object(beneficial_prop_gen_data, 'UvPley_Dim10_P2_Amito_PopBeneMutProp_N500_Fit05_SM005')
    save_object(beneficial_s_mean_gen_data, 'UvPley_Dim10_P2_Amito_BeneMutMeanS_N500_Fit05_SM005')
    save_object(s_mean_gen_data, 'UvPley_Dim10_P2_Amito_MutMeanS_N500_Fit05_SM005')
    save_object(s_abs_mean_gen_data, 'UvPley_Dim10_P2_Amito_MutAbsMeanS_N500_Fit05_SM005')

    save_object(t_mut_s_gen_data, 'UvPley_Dim10_P2_Amito_AllMutS_N500_Fit05_SM005')
    save_object(t_bene_mut_s_gen_data, 'UvPley_Dim10_P2_Amito_BeneGen_Mut_S_N500_Fit05_SM005')
    save_object(t_exist_mut_s_mean_gen_data, 'UvPley_Dim10_P2_Amito_AllExistMut_MEffect_N500_Fit05_SM005')
    save_object(t_exist_mut_abs_s_mean_gen_data, 'UvPley_Dim10_P2_Amito_AllExistMut_AbsMEffect_N500_Fit05_SM005')
    save_object(t_exist_mut_bene_prop_gen_data, 'UvPley_Dim10_P2_Amito_AllExistMut_Bene_Prop_N500_Fit05_SM005')    
    save_object(t_exist_mut_bene_s_mean_gen_data, 'UvPley_Dim10_P2_Amito_AllExistMut_BeneMEffect_N500_Fit05_SM005') 

    # print 'MEAN FIT', final

    end = time.time()
    print 'TOTAL TIME:', end-start